# AlignTune DPO

The Track 1 chain is CuratorKIT (data) to AlignTune (post-training) to AuditKIT (evaluation).
This kernel trains one DPO adapter through AlignTune's `create_rl_trainer`, after a 5-step smoke.

If the smoke fails, the kernel stops and the TRL path in `kaggle/train` remains the fallback.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path

def find_repo():
    hits = [p for p in Path("/kaggle/input").rglob("rule_checker.py")
            if p.exists() and (p.parents[2] / "scripts" / "train.py").exists()]
    hits.sort(key=lambda p: len(p.parts))
    if not hits:
        raise SystemExit("code dataset not mounted")
    return hits[0].parents[2]

WORK = Path("/kaggle/working/htd")
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(find_repo(), WORK)
os.chdir(WORK)
sys.path.insert(0, str(WORK / "src"))

def find_file(name):
    hits = [p for p in Path("/kaggle/input").rglob(name) if p.exists()]
    hits.sort(key=lambda p: len(p.parts))
    return hits[0] if hits else None

for p in Path("/kaggle/input").rglob("hf_token.txt"):
    os.environ.setdefault("HF_TOKEN", p.read_text().strip())
    os.environ.setdefault("HUGGINGFACE_API_KEY", p.read_text().strip())
assert os.environ.get("HF_TOKEN"), "attach rockerleo/htd-secrets"

DPO_DATA = find_file("dpo_train.jsonl")
print("DPO data:", DPO_DATA)
assert DPO_DATA, "attach rockerleo/htd-pairs"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

def run(cmd, **kw):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    return subprocess.run(cmd, check=True, **kw)

def assert_adapter(name):
    p = WORK / "outputs" / name
    cfg = p / "adapter_config.json"
    weights = p / "adapter_model.safetensors"
    assert cfg.exists(), f"{name}: adapter_config.json missing"
    assert weights.exists() and weights.stat().st_size > 1_000_000, f"{name}: adapter weights missing or empty"
    print(f"{name}: adapter ok")


In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "aligntune"])

In [ ]:
run(["aligntune", "info"], capture_output=True)

In [ ]:
run([sys.executable, "scripts/train_aligntune.py", "--data", str(DPO_DATA),
     "--output", "outputs/aligntune_smoke", "--max-steps", "5", "--grad-accum", "2"])
assert_adapter("aligntune_smoke")

In [ ]:
run([sys.executable, "scripts/train_aligntune.py", "--data", str(DPO_DATA),
     "--output", "outputs/aligntune_dpo", "--epochs", "1", "--lr", "5e-6", "--beta", "0.1", "--rank", "16"])
assert_adapter("aligntune_dpo")

In [ ]:
for name in ("aligntune_smoke", "aligntune_dpo"):
    p = WORK / "outputs" / name
    print(name, sorted(x.name for x in p.iterdir()) if p.exists() else "MISSING")